# JUG quick start

The core JUG Python API in six steps, on J1909-3744 (MeerKAT, 2828 TOAs).
The par file carries a noise model (EFAC, EQUAD, ECORR, power-law DM noise),
so `fit_parameters()` runs a generalised least-squares fit.

`J1909-3744.par` is a *deliberately perturbed* starting ephemeris: every
fitted parameter has been offset by 3 times its par-file uncertainty, so the
example shows the fit recovering a solution rather than starting at one.
Those par-file uncertainties are considerably larger than the fit's own
formal errors, so the offsets are tens of post-fit sigma and the pre-fit
residuals are correspondingly large.

## 0. Backend

JUG runs on whatever JAX backend is installed. This example pins the CPU
backend so it is reproducible anywhere; export `JAX_PLATFORMS=cuda` before
starting the kernel to run on a GPU instead.

In [1]:
import os

os.environ.setdefault("JAX_PLATFORMS", "cpu")  # must be set before JAX is imported

'cpu'

## 1. Open a session

In [2]:
from jug.engine.session import TimingSession

session = TimingSession("J1909-3744.par", "J1909-3744.tim")
print(session)

TimingSession(par='J1909-3744.par', tim='J1909-3744.tim', ntoas=2828)


## 2. Compute pre-fit residuals

In [3]:
pre = session.compute_residuals()
print(f"pre-fit RMS = {pre['rms_us']:.3f} us  ({pre['n_toas']} TOAs)")
print("free parameters:", session.free_params)

pre-fit RMS = 101.771 us  (2828 TOAs)
free parameters: ['A1', 'DECJ', 'DM1', 'DM2', 'EPS1', 'EPS2', 'F0', 'F1', 'FD1', 'FD2', 'FD3', 'FD4', 'FD5', 'FD6', 'FD7', 'FD8', 'FD9', 'M2', 'PB', 'PBDOT', 'PMDEC', 'PMRA', 'PX', 'RAJ', 'SINI', 'TASC', 'XDOT']


## 3. Fit the timing model

`fit_parameters()` fits every parameter flagged free in the par file and
auto-detects the noise model, so this is a GLS fit. Starting from the
perturbed ephemeris, the RMS should drop by more than two orders of
magnitude.

In [4]:
fit = session.fit_parameters()
print(f"post-fit RMS = {fit['final_rms']:.3f} us   chi2 = {fit['final_chi2']:.1f}")
print(f"{fit['iterations']} iterations, converged={fit['converged']}")

post-fit RMS = 0.497 us   chi2 = 12531.9
51 iterations, converged=True


Pre/post-fit parameter comparison with uncertainties:

In [5]:
session.parameter_table(fit)

Parameter                     Pre-fit               Post-fit    Uncertainty  Delta/sigma
----------------------------------------------------------------------------------------
RAJ                  5.01690709721721       5.01690710139597   2.943821e-10        14.20
DECJ               -0.658643174606078     -0.658643187124472   7.137430e-10        17.54
PMRA                -9.81596268690145      -9.52749449314923   1.935218e-02        14.91
PMDEC               -34.6383224452126      -35.7231281013277   6.592583e-02        16.45
F0                   339.315691919153       339.315691919041   7.967196e-13       139.70
F1              -1.61668395391594e-15  -1.61475826484323e-15   1.407852e-20       136.78
DM1              -0.00014496778640388   4.91950940258869e-05   5.164680e-05         3.76
DM2              2.53504682436982e-05   -5.4214943229156e-05   2.811456e-05         2.83
PX                   1.48273590236061      0.919723680257297   1.446967e-01         3.89
PB                   

## 4. Estimate the noise model (MAP)

Stochastic-parameter estimation by SVI at fixed timing model. The default
estimates EFAC, EQUAD, ECORR, red noise and DM noise.

In [6]:
est = session.estimate_noise()
for name, value in est.params.items():
    print(f"{name:<16} {value:>10.4f}")

EFAC_KAT_MKBF        1.0530
EQUAD_KAT_MKBF       0.0293
ECORR_KAT_MKBF       0.0920
TNREDAMP           -13.9922
TNREDGAM             1.8906
TNREDC              30.0000
TNDMAMP            -13.4716
TNDMGAM              1.7299
TNDMC               30.0000


Pass any estimator option through to choose what is estimated — e.g. white
noise and DM noise only, with a shorter SVI run:

In [7]:
white_dm = session.estimate_noise(include_red_noise=False, max_num_batches=10)
print(list(white_dm.params))

['EFAC_KAT_MKBF', 'EQUAD_KAT_MKBF', 'ECORR_KAT_MKBF', 'TNDMAMP', 'TNDMGAM', 'TNDMC']


## 5. Write the post-fit ephemeris

In [8]:
session.save_par("J1909-3744_postfit.par", fit_result=fit)
print(open("J1909-3744_postfit.par").read()[:400])

# Created by JUG on 2026-08-03 20:39:44
#
EPHEM                   DE440
CLK                     TT(BIPM2024)
UNITS                   TDB
TIMEEPH                 FB90
T2CMETHOD               IAU2000B
DILATEFREQ              N
NTOA                    2828.0
START                   58526.21388912177
FINISH                  60837.85782723828
PLANET_SHAPIRO          Y
CORRECT_TROPOSPHERE     Y
NE_SW   


## 6. Controlling which parameters are fitted

`set_frozen()` / `set_free()` edit the fit flags in memory, so the fitted
set can be changed without touching the par file. Freezing everything else
is how you fit a subset: `fit_parameters(fit_params=[...])` *adds* to the
par-file free parameters rather than replacing them.

In [9]:
sub = TimingSession("J1909-3744.par", "J1909-3744.tim")
sub.set_frozen([p for p in sub.free_params if p not in ("F0", "F1", "DM1")])
print("fitting:", sub.free_params)
print("design matrix:", sub.fit_parameters()["design_matrix_labels"])

fitting: ['DM1', 'F0', 'F1']


design matrix: ['OFFSET', 'DM1', 'F0', 'F1']


`set_free()` also works for a parameter the par file carries but does not
flag (`DM` here), and for a spin term the par file omits entirely (`F2`,
which starts from 0).

In [10]:
sub.set_free("DM", "F2")
sub_fit = sub.fit_parameters()
for name in ("DM", "F2"):
    print(f"{name:<4}{sub_fit['final_params'][name]:>18.9g}"
          f" +/- {sub_fit['uncertainties'][name]:.3g}")

DM          10.3490228 +/- 1.45e-05
F2      -1.2895305e-25 +/- 6.85e-28
